# UniRig trên Google Colab — Rig nhân vật 1 bằng AI

**Trước khi chạy:** vào menu Colab **Runtime → Change runtime type → Hardware accelerator → T4 GPU** rồi bấm **Connect**. Không có GPU thì UniRig không chạy được (nó cần CUDA, VRAM tối thiểu 8 GB).

Notebook này rig **nhân vật 1** (`tripo_convert_nhanvat1.glb`, Tripo, 50.000 tris, chưa có xương) đúng 3 bước của UniRig:
1. **Skeleton** — AI đoán xương cho nhân vật (`generate_skeleton.sh`)
2. **Skinning** — AI tính trọng số skin cho từng đỉnh (`generate_skin.sh`)
3. **Merge** — gộp xương + skin vào model gốc → file đã rig (`merge.sh`)

Checkpoint tự tải từ Hugging Face `VAST-AI/UniRig` khi chạy. Lần chạy đầu cài đặt mất khoảng 10–20 phút.


In [ ]:
# 1. Kiểm tra GPU — phải thấy tên GPU (VD: Tesla T4) và CUDA, nếu không thì đổi runtime type
!nvidia-smi
!python3 --version
import torch
print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

In [ ]:
# 2. Lấy code UniRig (bản fork của bạn trên GitHub)
!git clone https://github.com/nguyenvu16592bb/UniRig.git
%cd UniRig
!git log --oneline -1
!ls examples/

In [ ]:
%%bash
# 3. Cài thư viện — Colab hiện chạy Python 3.13 nhưng UniRig chỉ chạy được trên Python 3.11,
# nên cell này tự tạo môi trường Python 3.11 riêng rồi cài đúng bộ thư viện CUDA cho nó.
# Cell này chạy lại được nhiều lần: bước nào xong rồi sẽ bỏ qua, chỉ làm tiếp phần còn thiếu.
# Lần đầu mất khoảng 5–10 phút (không còn bước biên dịch).
# LƯU Ý: `uv venv` không tự tạo pip trong môi trường, nên mọi lệnh cài đều dùng
# `uv pip install --python ...` thay vì gọi pip trong venv.
# Cả bộ dùng CUDA 11.8 (không dùng 12.1) vì flash-attn 2.6.3 không có wheel cho torch 2.4 + cu121
# (chỉ có cu118/cu123), và spconv-cu121 không có bản 2.3.6 (chỉ có 2.3.7/2.3.8).
# Torch dùng 2.3.1. torch-scatter dùng WHEEL LÀM SẴN pt23 (cho torch 2.3.1+cu118):
# đã kiểm chứng import OK trong bản Gemini lưu ở 3D-Model (torch-scatter 2.1.2+pt23cu118).
# LƯU Ý QUAN TRỌNG: KHÔNG biên dịch torch-scatter từ source trên Colab nữa — nvcc 11.8
# không đọc được header glibc 2.39 của Ubuntu 24.04 (lỗi `__builtin_dynamic_object_size`
# và `_Float32 is undefined`, 100 errors ở scatter_cuda.cu); NVCC_FLAGS cũng không sửa được.
# Vì dùng wheel nên KHÔNG cần cài CUDA Toolkit / nvcc nữa — torch đã có sẵn thư viện CUDA.
# Chỉ wheel pt24 (torch 2.4.x) mới lỗi `undefined symbol: parseSchemaOrName` — đừng dùng.
set -e
export PATH="$HOME/.local/bin:$PATH"
if [ ! -x "$HOME/.local/bin/uv" ]; then
  curl -LsSf https://astral.sh/uv/install.sh | sh
  export PATH="$HOME/.local/bin:$PATH"
fi
uv python install 3.11
if [ ! -x /content/venv311/bin/python ]; then
  uv venv /content/venv311 --python 3.11
fi
V=/content/venv311/bin/python
uv pip install --python $V "torch==2.3.1+cu118" --index-url https://download.pytorch.org/whl/cu118
uv pip install --python $V spconv-cu118==2.3.8
uv pip install --python $V https://github.com/Dao-AILab/flash-attention/releases/download/v2.6.3/flash_attn-2.6.3+cu118torch2.3cxx11abiFALSE-cp311-cp311-linux_x86_64.whl
uv pip install --python $V setuptools wheel ninja
if ! $V -c "import torch_scatter" 2>/dev/null; then
  uv pip uninstall --python $V torch-scatter 2>/dev/null || true
  uv pip install --python $V torch-scatter -f https://data.pyg.org/whl/torch-2.3.1+cu118.html
fi
uv pip install --python $V -r requirements.txt
$V -c "import torch, spconv, torch_scatter, flash_attn, bpy; print('OK - torch', torch.__version__, '| cuda:', torch.cuda.is_available(), '| bpy OK')" 


In [ ]:
# Dùng môi trường Python 3.11 vừa cài ở cell 3 cho mọi lệnh sau này
import os
os.environ['PATH'] = '/content/venv311/bin:' + os.environ['PATH']
# 4. File nhân vật 1 — có sẵn trong repo sau khi clone, không cần up lại
INPUT = 'examples/tripo_convert_nhanvat1.glb'
NAME = 'nhanvat1'
if os.path.exists(INPUT):
    print('Dùng file nhân vật 1 có sẵn trong repo:', INPUT)
else:
    print('Không thấy file trong repo — up file .glb/.fbx/.obj của bạn ở ô hiện ra:')
    from google.colab import files
    uploaded = files.upload()
    INPUT = list(uploaded.keys())[0]
    NAME = INPUT.rsplit('.', 1)[0].replace(' ', '_')
    print('Input:', INPUT)
# Đưa INPUT/NAME sang biến môi trường để các cell bash (5-7) dùng được ($INPUT, $NAME)
os.environ['INPUT'] = INPUT
os.environ['NAME'] = NAME
print('Tên file xuất ra sẽ bắt đầu bằng:', NAME)



In [ ]:
%%bash
# 5. Bước 1 — AI đoán xương cho nhân vật 1 (checkpoint tự tải từ Hugging Face lần đầu)
# Dùng --output_dir thay cho --output: file xương sẽ ở results/$NAME/skeleton.fbx.
# (Cách --output cũ không sinh ra file mà cũng không báo lỗi — đã gặp ở bản Gemini.)
set -e
export PATH="/content/venv311/bin:$PATH"
cd /content/UniRig
if [ ! -f "./$NAME.glb" ]; then cp "$INPUT" "./$NAME.glb"; fi
mkdir -p results
SKEL="results/$NAME/skeleton.fbx"
if [ ! -f "$SKEL" ]; then
  echo "=== Bước 1: Đoán xương ==="
  bash launch/inference/generate_skeleton.sh --input "./$NAME.glb" --output_dir results
  [ -f "$SKEL" ] || { echo "LỖI BƯỚC 1: không có $SKEL. Nội dung tmp/$NAME:"; ls -lh "tmp/$NAME/"; exit 1; }
else
  echo "Bước 1 đã có sẵn: $SKEL"
fi
ls -lh "$SKEL"



In [ ]:
%%bash
# 6. Bước 2 — AI tính skinning cho nhân vật 1 (dựa trên xương vừa đoán; nếu xương sai thì sửa xương trước rồi mới chạy bước này)
set -e
export PATH="/content/venv311/bin:$PATH"
cd /content/UniRig
SKIN="results/${NAME}_skin.fbx"
if [ ! -f "$SKIN" ]; then
  echo "=== Bước 2: Tính skinning ==="
  bash launch/inference/generate_skin.sh --input "results/$NAME/skeleton.fbx" --output "$SKIN"
  [ -f "$SKIN" ] || { echo "LỖI BƯỚC 2: không có $SKIN"; ls -lh results/; exit 1; }
else
  echo "Bước 2 đã có sẵn: $SKIN"
fi
ls -lh "$SKIN"



In [ ]:
%%bash
# 7. Bước 3 — Gộp xương + skin vào nhân vật 1 → file đã rig hoàn chỉnh
set -e
export PATH="/content/venv311/bin:$PATH"
cd /content/UniRig
RIGGED="results/${NAME}_rigged.glb"
if [ ! -f "$RIGGED" ]; then
  echo "=== Bước 3: Gộp thành file rig ==="
  bash launch/inference/merge.sh --source "results/${NAME}_skin.fbx" --target "./$NAME.glb" --output "$RIGGED"
  [ -f "$RIGGED" ] || { echo "LỖI BƯỚC 3: không có $RIGGED"; ls -lh results/; exit 1; }
else
  echo "Bước 3 đã có sẵn: $RIGGED"
fi
echo "=== XONG ==="
ls -lh results/



In [ ]:
# 8. Tải file nhân vật 1 đã rig về máy
from google.colab import files
files.download(f'results/{NAME}_rigged.glb')
files.download(f'results/{NAME}/skeleton.fbx')



## Nếu gặp lỗi

- **Cell 3 báo `bpy==4.2` không có bản phù hợp / `torch_scatter` / `flash_attn` build lỗi** → đó là vì Colab hiện chạy Python 3.13, UniRig chỉ chạy trên Python 3.11. Cell 3 bản mới tự tạo môi trường Python 3.11 riêng (uv) và cài torch 2.3.1+cu118 + wheel torch-scatter pt23 + spconv-cu118 2.3.8 + flash-attn 2.6.3 + bpy 4.2 — chạy lại cell 3 rồi chạy tiếp cell 4–8. Đừng cài thêm gì bằng pip ngoài cell 3. (Không biên dịch torch-scatter từ source: nvcc 11.8 không đọc được header glibc của Ubuntu 24.04.)
- **Không thấy GPU / CUDA available: False** → Runtime → Change runtime type → chọn T4 GPU, rồi chạy lại từ cell 1.
- **Một bước báo `LỖI BƯỚC 1/2/3: không có file...`** → dừng lại, đừng chạy tiếp: chụp ảnh toàn bộ output của cell đó gửi mình. File xương nằm ở `results/nhanvat1/skeleton.fbx`, file skin ở `results/nhanvat1_skin.fbx`, file rig ở `results/nhanvat1_rigged.glb`.
- **Hết VRAM (CUDA out of memory)** → model đầu vào quá nặng: giảm mặt model xuống dưới 50.000 tris trước khi đưa vào (UniRig tự decimate về 50.000 nhưng model gốc quá lớn vẫn có thể tràn).
- **Muốn rig model khác** → sửa `INPUT` ở cell 4 thành file bạn upload (hoặc xóa file nhân vật 1 khỏi repo rồi up file mới ở cell 4), rồi chạy lại cell 5–7 — tên file xuất ra đổi theo `NAME` tự động.

Repo gốc: https://github.com/VAST-AI-Research/UniRig — Paper SIGGRAPH 2025.

